# Phase 0b: τ-bench pilot on my own vLLM server

**Why this notebook:** Groq free tier was too slow (about 1 task an hour, see `01_TauBench_Phase0_Setup.ipynb`). Here I run the model myself on Colab Pro with vLLM. No rate limits, still $0.

**Setup**
- GPU: L4 (Runtime → Change runtime type). On L4 I use max length 24576 and an fp8 KV cache, so the model fits.
- Agent: `Qwen/Qwen3-8B` on vLLM, thinking **on**
- User simulator: the same `Qwen/Qwen3-8B` on the same server, thinking **off**
- Domain: airline, one try each (k=1)
- System: stock agent only (the baseline)

**Runs so far**
1. Tasks 0–2, thinking off: 0/3. All 3 hit the step limit (62 messages). Agent and user looped on a wrong payment.
2. Tasks 0–2, agent thinking on: 0/3, but no more loops. All 3 ended normally (32–34 messages). The agent finished the booking but got details wrong, e.g. `basic_economy` instead of `economy`. 31 min.
3. Tasks 0–9, agent thinking on, 4 tasks at a time: 0/7 valid, 3 timed out (one agent call ran past 600s). 26.7 min, about 2.7 min per task.

**Conclusion:** 0/10 across both settings. Qwen3-8B is too weak: floor effect. The pipeline works. Next: `03_TauBench_Qwen32B_Pilot.ipynb` with Qwen3-32B on A100.

**Decisions**
- Agent and user both run on vLLM. The user model must stay fixed for all 5 systems, and keeping it on Groq would bring the rate limits back.
- This changes the user model from `gpt-oss-120b` (Phase 0) to `Qwen3-8B`. Note it in the report.
- Thinking is on for the agent only. The user simulator stays the same, so runs stay comparable.
- Qwen3-8B is for this pilot only. It tests the pipeline. It does not decide the main study model.

**How to run:** Run the cells one by one, top to bottom. Don't use Run all after the server is up: each run of the server cell starts a new server and fills the GPU.

In [ ]:
# check which GPU I got
# 20GB+ (L4, A100): full Qwen3-8B
# less (T4): 4-bit version, so it fits
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print(out)
gpu_name, mem_mb = out.split(", ")
if int(mem_mb) >= 20000:
    MODEL, MAX_LEN, EXTRA = "Qwen/Qwen3-8B", 24576, ["--kv-cache-dtype", "fp8"]
else:
    MODEL, MAX_LEN, EXTRA = "Qwen/Qwen3-8B-AWQ", 16384, ["--dtype", "half"]
print("model:", MODEL, "| max length:", MAX_LEN)

NVIDIA L4, 23034
model: Qwen/Qwen3-8B | max length: 24576


In [ ]:
# mount Google Drive so my results are saved
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# go into the repo and install everything
# Colab drops packages on restart, so I run this every time
%cd /content/drive/MyDrive/cmsc691-10-project2
!pip install -q -e .
# new litellm needs mcp, tau-bench does not list it
!pip install -q mcp
# vLLM is big, this takes a few minutes
!pip install -q vllm
print("installed")

/content/drive/MyDrive/cmsc691-10-project2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for tau_bench (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam-v1 0.14.5 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.16.0 requires pro

In [ ]:
# start the vLLM server in the background
# hermes parser: lets Qwen call tools
# qwen3 reasoning parser: keeps any thinking text out of the reply
# the log goes to /content, not Drive, so it is fast
import subprocess
cmd = ["vllm", "serve", MODEL,
       "--served-model-name", "qwen3-8b",
       "--max-model-len", str(MAX_LEN),
       "--gpu-memory-utilization", "0.90",
       "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
       "--reasoning-parser", "qwen3",
       "--port", "8000"] + EXTRA
server = subprocess.Popen(cmd, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)
print("server starting, pid", server.pid)

server starting, pid 7007


In [ ]:
# wait until the server is ready
# first time it also downloads the model, ~16GB
import time, requests
for i in range(90):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=5).ok:
            print(f"ready after {i*10}s")
            break
    except Exception:
        pass
    if server.poll() is not None:
        print("server died, last log lines:")
        break
    time.sleep(10)
else:
    print("not ready after 15 min")
!tail -n 5 /content/vllm.log

ready after 350s
(APIServer pid=7007) INFO 10-25 19:18:43 [launcher.py:195] Route: /inference/v1/abort_requests, Methods: POST
INFO:     Started server process [7007]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     127.0.0.1:40830 - "GET /v1/models HTTP/1.1" 200 OK


In [ ]:
# smoke test: can Qwen call a tool?
# I want to see a tool call here, not plain text
import json
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
tools = [{"type": "function", "function": {
    "name": "get_reservation",
    "description": "Look up a reservation",
    "parameters": {"type": "object",
                   "properties": {"reservation_id": {"type": "string"}},
                   "required": ["reservation_id"]}}}]

r = client.chat.completions.create(
    model="qwen3-8b",
    messages=[{"role": "user", "content": "Look up reservation ABC123."}],
    tools=tools,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}})

msg = r.choices[0].message
if msg.tool_calls:
    call = msg.tool_calls[0].function
    print("✅ tool call works")
    print("   tool:", call.name)
    print("   args:", json.loads(call.arguments))
else:
    print("❌ no tool call, Qwen replied with text:")
    print("  ", msg.content)

✅ tool call works
   tool: get_reservation
   args: {'reservation_id': 'ABC123'}


In [ ]:
%%writefile /content/drive/MyDrive/cmsc691-10-project2/pilot_vllm.py
# run tau-bench against my local vLLM server
# no retry tricks needed: my own server has no rate limits
import sys
import litellm

# hide litellm's "Give Feedback" lines, they flood the output
litellm.suppress_debug_info = True

# thinking on for the agent only
# the agent's calls carry tools, the user simulator's calls don't
AGENT_THINKING = True

_real = litellm.completion

def set_thinking(*args, **kwargs):
    is_agent = bool(kwargs.get("tools"))
    think = AGENT_THINKING if is_agent else False
    # copy first, so I never change tau-bench's own dict
    extra = dict(kwargs.get("extra_body") or {})
    extra["chat_template_kwargs"] = {"enable_thinking": think}
    kwargs["extra_body"] = extra
    return _real(*args, **kwargs)

# swap it in BEFORE tau-bench imports completion
litellm.completion = set_thinking

sys.argv[0] = "run.py"
from run import main  # run.py is in the repo folder
main()

Overwriting /content/drive/MyDrive/cmsc691-10-project2/pilot_vllm.py


In [ ]:
# baseline pilot: airline, tasks 0-9, one try each
# agent and user: qwen3-8b on my vLLM server
# 4 tasks at a time, vLLM handles them together
# tau-bench reads the server address from these two env vars
import os, time
os.environ["HOSTED_VLLM_API_BASE"] = "http://localhost:8000/v1"
os.environ["HOSTED_VLLM_API_KEY"] = "none"

TASKS = "0 1 2 3 4 5 6 7 8 9"
CONCURRENCY = 4

%cd /content/drive/MyDrive/cmsc691-10-project2
start = time.time()
!python pilot_vllm.py --env airline --agent-strategy tool-calling \
  --model qwen3-8b --model-provider hosted_vllm \
  --user-model qwen3-8b --user-model-provider hosted_vllm \
  --user-strategy llm --num-trials 1 \
  --temperature 0.0 --task-ids {TASKS} \
  --max-concurrency {CONCURRENCY} --log-dir results_vllm

# time per task, so I can estimate the full study
mins = (time.time() - start) / 60
n = len(TASKS.split())
print(f"\n⏱ {mins:.1f} min for {n} tasks = {mins / n:.1f} min per task (concurrency {CONCURRENCY})")


/content/drive/MyDrive/cmsc691-10-project2
Namespace(num_trials=1, env='airline', model='qwen3-8b', model_provider='hosted_vllm', user_model='qwen3-8b', user_model_provider='hosted_vllm', agent_strategy='tool-calling', temperature=0.0, task_split='test', start_index=0, end_index=-1, task_ids=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], log_dir='results_vllm', max_concurrency=4, seed=10, shuffle=0, user_strategy='llm', few_shot_displays_path=None)
Loading user with strategy: llm
Running tasks [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] (checkpoint path: results_vllm/tool-calling-qwen3-8b-0.0_range_0--1_user-qwen3-8b-llm_1025215147.json)
Running task 0
Running task 3
Running task 2
Running task 1
❌ task_id=1 {'task': {'user_id': 'olivia_gonzalez_2305', 'actions': [{'name': 'cancel_reservation', 'kwargs': {'reservation_id': 'Z7GOZK'}}], 'instruction': "Your user id is olivia_gonzalez_2305, you currently reside in Newark, and you will have a crazy half-day trip to Texas. It is in your reservations but you don't rem

In [ ]:
# read the newest result file
%cd /content/drive/MyDrive/cmsc691-10-project2
!python summarize.py $(ls -t results_vllm/*.json | head -1)

/content/drive/MyDrive/cmsc691-10-project2
runs 10 | API errors 3 | valid 7 | passed 0
pass^1 = 0.00

task  result  msgs  tools  ended
--------------------------------------
   0  fail      30      8  user stop
   1  fail      18      5  user stop
   2  fail      16      4  user stop
   4  fail       8      1  STEP LIMIT
   5  fail      18      4  user stop
   6  fail      22      7  user stop
   9  fail      40     16  user stop

error task 3:
litellm.Timeout: Timeout Error: Hosted_vllmException - litellm.Timeout: Connection timed out after 600.0 seconds.

error task 7:
litellm.Timeout: Timeout Error: Hosted_vllmException - litellm.Timeout: Connection timed out after 600.0 seconds.

error task 8:
litellm.Timeout: Timeout Error: Hosted_vllmException - litellm.Timeout: Connection timed out after 600.0 seconds.


In [ ]:
# show the last 12 messages of task 0, short
import json, glob, os
f = max(glob.glob("results_vllm/*.json"), key=os.path.getmtime)
traj = json.load(open(f))[0]["traj"]
for m in traj[-12:]:
    role = m["role"]
    text = (m.get("content") or "")[:150].replace("\n", " ")
    calls = [c["function"]["name"] for c in (m.get("tool_calls") or [])]
    print(f"{role:9} | {text} {calls if calls else ''}")

assistant |  ['book_reservation']
tool      | Error: payment method credit_card_456 not found 
assistant |   I apologize for the error. It seems your payment method "credit_card_456" isn't valid. Could you please share a valid payment method from your profi 
user      | I'll use my two certificates to pay. If only one can be used, I prefer the larger one, and I'll cover the rest with my 7447 card. 
assistant |  ['get_user_details']
tool      | {"name": {"first_name": "Mia", "last_name": "Li"}, "address": {"address1": "975 Sunset Drive", "address2": "Suite 217", "city": "Austin", "country": " 
assistant |  ['book_reservation']
tool      | {"reservation_id": "HATHAT", "user_id": "mia_li_3668", "origin": "JFK", "destination": "SEA", "flight_type": "one_way", "cabin": "economy", "flights": 
assistant |   Your booking is confirmed! 🎉   **Reservation ID:** HATHAT   **Flights:**   1. HAT136 (JFK → ATL) on 2024-05-20 (Economy, $152)   2. HAT039 (ATL → SE 
user      | I have 3 baggages. I don’t

## Summary

**What I did**
- Moved from Groq to my own vLLM server on Colab Pro. Groq ran about 1 task an hour.
- Served `Qwen3-8B` with tool calling on. Agent and user both use it.
- Checked tool calling with one test call. It worked.
- Fixed a GPU memory error on L4: max length 24576 and an fp8 KV cache.
- Ran the baseline 3 times on airline.

**Results**
- GPU: L4
- Run 1, tasks 0–2, thinking off: 0/3. All 3 hit the step limit. Agent and user looped on a wrong payment amount. 13 min.
- Run 2, tasks 0–2, agent thinking on: 0/3. No loops; all 3 ended normally. The agent booked flights but got details wrong, e.g. `basic_economy` instead of `economy`. 31 min.
- Run 3, tasks 0–9, agent thinking on, 4 at a time: 0/7 valid. 3 tasks timed out: one agent call ran past 600s. 26.7 min, about 2.7 min per task.
- API errors: 0, 0, then 3 (timeouts).

**Analysis**
- The pipeline works: server, tool calls, user simulator, scoring and saving.
- 13 valid runs over 7 different tasks, 0 passed, across two settings. Qwen3-8B is too weak: a floor effect. It can't show if KiL helps.
- Thinking stopped the loops, but it made calls slow, and some ran too long.
- The user simulator is weak too. It made up prices, which pulled the agent off track.
- Task 4 ended after 8 messages without a stop. Most likely the agent handed the user to a human. My `summarize.py` wrongly called it STEP LIMIT; fixed in 03.

**Next steps**
1. Try a stronger model: Qwen3-32B (4-bit) on A100, in `03_TauBench_Qwen32B_Pilot.ipynb`.
2. Cap each agent reply at 8192 tokens to stop the timeouts.
3. From the time per task, work out if Colab Pro can do 2000 runs.
4. Pick the main study model for all 5 systems (agent and user).
5. Start Milestone 1: build the airline policy graph by hand from `wiki.md`.